# HakaPokes — Pipeline de Datos Sintéticos y Modelado Predictivo
### TFM Equipo 4A — Entregable 3 (notebook de verificación / alineación de resultados)

Este notebook implementa la metodología descrita en el documento del proyecto:
generación de ~500,000 registros transaccionales sintéticos, y entrenamiento de los
modelos documentados para los dos módulos analíticos:

- **Módulo 1:** predicción de demanda (regresión) — Ridge, Random Forest, HistGradientBoosting — split temporal 80/20.
- **Módulo 2:** propensión a venta cruzada de bebidas (clasificación) — Random Forest estándar vs. balanceado, Regresión Logística — split estratificado 80/20.

> **Nota importante:** este dataset es sintético y generado con supuestos propios (no es
> el dataset original de HakaPokes ni el que produjo las cifras ya escritas en el documento
> o en el dashboard inicial de la Entrega anterior). Su propósito es proveer una **plantilla de código ejecutable y
> metodológicamente fiel** al TFM. Las métricas que arroja esta ejecución **no deben
> copiarse tal cual** al documento — deben sustituirse por las que resulten de correr este
> mismo pipeline sobre el dataset sintético real ajustado,
> y esas cifras (una sola versión) son las que están reflejadas de forma consistente
> en el documento Word, las tablas-imagen (Tabla 3 y Tabla 4) y el dashboard.


In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, HistGradientBoostingRegressor
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                              roc_auc_score, recall_score, precision_score, f1_score,
                              confusion_matrix, accuracy_score)
from sklearn.model_selection import train_test_split

RNG_SEED = 42
rng = np.random.default_rng(RNG_SEED)
N = 500_000
print("Semilla:", RNG_SEED, "| Registros a generar:", N)


Semilla: 42 | Registros a generar: 500000


## 1. Generación de Datos Sintéticos

Se replican las reglas de negocio documentadas: 5 sucursales (Campanario, Juriquilla,
Refugio, Centro Sur, Jurica), tres tamaños de poke bowl (Chico/Mediano/Grande) con precio
base propio, horario operativo de 11:00 a 22:00 con concentración en horas pico
(13:00–15:00 y 18:00–20:00), estacionalidad por fin de semana/quincena, y una variable de
temperatura ambiente simulada (estacionalidad anual + ruido), tal como se describe en la
sección "Generación de Datos Sintéticos" del documento.


In [2]:
sucursales = ["Campanario", "Juriquilla", "Refugio", "Centro Sur", "Jurica"]
sucursal_probs = [0.24, 0.22, 0.20, 0.18, 0.16]

tamanos = ["Chico", "Mediano", "Grande"]
tamano_probs = [0.30, 0.45, 0.25]
tamano_num_map = {"Chico": 1, "Mediano": 2, "Grande": 3}
precio_base = {"Chico": 120.0, "Mediano": 155.0, "Grande": 195.0}

start_date = pd.Timestamp("2025-01-06")
total_days = 430
day_offsets = rng.integers(0, total_days, size=N)
fechas = start_date + pd.to_timedelta(day_offsets, unit="D")
dia_semana_num = fechas.dayofweek.values

horas_posibles = np.arange(11, 23)
peso_horas = np.array([0.05, 0.07, 0.14, 0.16, 0.14, 0.06, 0.05, 0.06, 0.08, 0.13, 0.14, 0.10])
peso_horas = peso_horas / peso_horas.sum()
hora = rng.choice(horas_posibles, size=N, p=peso_horas)
es_hora_pico = np.isin(hora, [13, 14, 15, 18, 19, 20]).astype(int)

sucursal = rng.choice(sucursales, size=N, p=sucursal_probs)
tamano = rng.choice(tamanos, size=N, p=tamano_probs)
tamano_num = np.vectorize(tamano_num_map.get)(tamano)

day_of_year = fechas.dayofyear.values
temp_estacional = 22 + 6 * np.sin((day_of_year / 365) * 2 * np.pi - np.pi / 2)
temperatura = temp_estacional + rng.normal(0, 3, size=N)

es_finde = np.isin(dia_semana_num, [5, 6]).astype(int)
es_quincena = np.isin(fechas.day.values, [15, 16, 29, 30, 31, 1]).astype(int)

ruido = rng.normal(0, 18, size=N)
precio = np.vectorize(precio_base.get)(tamano)
venta_efectiva = (precio + 25*es_hora_pico + 15*es_finde + 10*es_quincena
                  + 0.8*(temperatura - 22) + ruido)
venta_efectiva = np.clip(venta_efectiva, 40, None)

logit = (-1.6 + 0.55*(tamano_num - 1) + 0.5*es_hora_pico
         + 0.06*(temperatura - 22) + 0.15*(dia_semana_num >= 5))
prob_bebida = 1 / (1 + np.exp(-logit))
compra_bebida = rng.binomial(1, prob_bebida)

df = pd.DataFrame({
    "fecha_registro": fechas, "dia_semana_num": dia_semana_num, "hora": hora,
    "es_hora_pico": es_hora_pico, "nombre_sucursal": sucursal, "tamano": tamano,
    "tamano_num": tamano_num, "temperatura": temperatura, "es_finde": es_finde,
    "es_quincena": es_quincena, "venta_efectiva": venta_efectiva, "compra_bebida": compra_bebida,
})

print("Shape:", df.shape)
print(df.head(5).to_string())
print("\nTasa de conversión a combo (compra_bebida=1):", round(df["compra_bebida"].mean(), 4))


Shape: (500000, 12)
  fecha_registro  dia_semana_num  hora  es_hora_pico nombre_sucursal   tamano  tamano_num  temperatura  es_finde  es_quincena  venta_efectiva  compra_bebida
0     2025-02-13               3    12             0      Campanario    Chico           1    12.474607         0            0      125.038647              0
1     2025-12-04               3    22             0      Campanario    Chico           1    19.131185         0            0      113.109650              0
2     2025-10-14               1    20             1         Refugio  Mediano           2    27.485773         0            0      169.762701              0
3     2025-07-13               6    21             0          Jurica   Grande           3    23.058773         1            0      190.957761              1
4     2025-07-11               4    16             0          Jurica  Mediano           2    26.381447         0            0      135.425935              0

Tasa de conversión a combo (compra_be

## 2. Módulo 1 — Predicción de Demanda (Regresión)

Se agregan las transacciones a nivel **hora × sucursal × día**, replicando el enfoque de
"predicción de ventas por hora" descrito en el documento, y se aplica una **separación
temporal 80/20** (los registros cronológicamente más recientes forman el conjunto de
prueba), evitando fuga de información hacia el pasado.


In [3]:
agg = (df.sort_values("fecha_registro")
         .groupby(["fecha_registro", "nombre_sucursal", "hora"], as_index=False)
         .agg(venta_hora=("venta_efectiva", "sum"),
              dia_semana_num=("dia_semana_num", "first"),
              es_hora_pico=("es_hora_pico", "first"),
              temperatura=("temperatura", "mean"),
              es_finde=("es_finde", "first"),
              es_quincena=("es_quincena", "first")))
agg = agg.sort_values("fecha_registro").reset_index(drop=True)
agg = pd.get_dummies(agg, columns=["nombre_sucursal"], drop_first=True)

feature_cols_m1 = [c for c in agg.columns if c not in ["fecha_registro", "venta_hora"]]
X, y = agg[feature_cols_m1], agg["venta_hora"]

split_idx = int(len(agg) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]
print(f"Registros agregados: {len(agg):,} | Train: {len(X_train):,} | Test: {len(X_test):,}")

modelos_m1 = {
    "Ridge Regression (Baseline)": Ridge(alpha=1.0, random_state=RNG_SEED),
    "Random Forest Regressor": RandomForestRegressor(n_estimators=200, max_depth=12, random_state=RNG_SEED, n_jobs=-1),
    "HistGradientBoostingRegressor": HistGradientBoostingRegressor(random_state=RNG_SEED),
}

resultados_m1 = []
print(f"\n{'Modelo':35s} | {'MAE':>10s} | {'RMSE':>10s} | {'R2':>8s}")
print("-"*70)
for nombre, modelo in modelos_m1.items():
    modelo.fit(X_train, y_train)
    pred = modelo.predict(X_test)
    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2 = r2_score(y_test, pred)
    resultados_m1.append((nombre, mae, rmse, r2))
    print(f"{nombre:35s} | ${mae:9.2f} | ${rmse:9.2f} | {r2:8.4f}")


Registros agregados: 25,799 | Train: 20,639 | Test: 5,160

Modelo                              |        MAE |       RMSE |       R2
----------------------------------------------------------------------
Ridge Regression (Baseline)         | $  1106.85 | $  1392.69 |   0.3541
Random Forest Regressor             | $   625.91 | $   811.90 |   0.7805
HistGradientBoostingRegressor       | $   604.82 | $   779.46 |   0.7977


## 3. Módulo 2 — Clasificación de Propensión a Venta Cruzada (Bebidas)

Se aplica una **separación estratificada 80/20** sobre la variable objetivo binaria
(`compra_bebida`), para preservar la misma tasa de conversión en entrenamiento y prueba,
y se compara el Random Forest estándar contra su versión con `class_weight='balanced'`,
además de una Regresión Logística balanceada como referencia adicional.


In [4]:
df_m2 = pd.get_dummies(df, columns=["nombre_sucursal"], drop_first=True)
feature_cols_m2 = ["dia_semana_num", "hora", "es_hora_pico", "tamano_num",
                    "temperatura", "es_finde", "es_quincena"] + \
                   [c for c in df_m2.columns if c.startswith("nombre_sucursal_")]

X2, y2 = df_m2[feature_cols_m2], df_m2["compra_bebida"]
X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.2, stratify=y2, random_state=RNG_SEED)

print(f"Train: {len(X2_train):,} | Test: {len(X2_test):,} | Tasa combo (train): {y2_train.mean():.4f}")

modelos_m2 = {
    "Random Forest (Estandar)": RandomForestClassifier(n_estimators=200, max_depth=10, random_state=RNG_SEED, n_jobs=-1),
    "Random Forest (balanced)": RandomForestClassifier(n_estimators=200, max_depth=10, class_weight="balanced", random_state=RNG_SEED, n_jobs=-1),
    "Regresion Logistica (balanced)": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RNG_SEED),
}

print(f"\n{'Modelo':32s} | {'ROC-AUC':>8s} | {'Recall(0)':>10s} | {'Prec(0)':>8s} | {'F1-macro':>9s} | {'Acc':>7s}")
print("-"*90)
cm = None
feat_importance_final = None
for nombre, modelo in modelos_m2.items():
    modelo.fit(X2_train, y2_train)
    pred = modelo.predict(X2_test)
    proba = modelo.predict_proba(X2_test)[:, 1]
    roc = roc_auc_score(y2_test, proba)
    rec = recall_score(y2_test, pred, pos_label=0)
    prec = precision_score(y2_test, pred, pos_label=0)
    f1m = f1_score(y2_test, pred, average="macro")
    acc = accuracy_score(y2_test, pred)
    print(f"{nombre:32s} | {roc:8.4f} | {rec:10.4f} | {prec:8.4f} | {f1m:9.4f} | {acc:7.4f}")
    if nombre == "Random Forest (balanced)":
        cm = confusion_matrix(y2_test, pred)
        feat_importance_final = pd.Series(modelo.feature_importances_, index=feature_cols_m2).sort_values(ascending=False)

print("\nMatriz de confusion (Random Forest balanced) [filas=real 0/1 , cols=pred 0/1]:")
print(cm)
print("\nImportancia de variables (Random Forest balanced), Modulo 2:")
print(feat_importance_final.round(4).to_string())


Train: 400,000 | Test: 100,000 | Tasa combo (train): 0.3266

Modelo                           |  ROC-AUC |  Recall(0) |  Prec(0) |  F1-macro |     Acc
------------------------------------------------------------------------------------------
Random Forest (Estandar)         |   0.6546 |     0.9494 |   0.6944 |    0.5125 |  0.6845
Random Forest (balanced)         |   0.6545 |     0.6358 |   0.7613 |    0.5982 |  0.6205
Regresion Logistica (balanced)   |   0.6560 |     0.6128 |   0.7659 |    0.5948 |  0.6131

Matriz de confusion (Random Forest balanced) [filas=real 0/1 , cols=pred 0/1]:
[[42815 24526]
 [13422 19237]]

Importancia de variables (Random Forest balanced), Modulo 2:
tamano_num                    0.4214
temperatura                   0.3463
es_hora_pico                  0.1346
hora                          0.0518
dia_semana_num                0.0205
es_finde                      0.0067
es_quincena                   0.0063
nombre_sucursal_Juriquilla    0.0033
nombre_sucursal_Cen

## 4. Conclusión metodológica

Los números que arroja **esta** ejecución (arriba) son distintos tanto de la "Tabla 3 /
Tabla 4" del documento como de los que aparecen en el dashboard — y **eso es esperado**:
el dataset sintético, el nivel de agregación del target y los hiperparámetros exactos no
son idénticos a los que usó el equipo originalmente. Este notebook no busca reproducir un
número específico "de memoria", sino dar una base de código auditable y reproducible.

**Siguiente paso recomendado:** ejecutar este mismo pipeline (o el notebook original del
equipo, si aún se conserva) sobre el dataset sintético real que ya generaron para el
proyecto, y tomar esa salida —una sola vez— como la cifra oficial. Esa cifra, y solo esa,
debe quedar reflejada de manera idéntica en:

1. El párrafo "Evaluación" del documento (Módulo 1 y Módulo 2).
2. Las tablas-imagen Tabla 3 y Tabla 4 (deben regenerarse como nueva captura de pantalla).
3. La sección "Alineación de Hallazgos con los Objetivos del Proyecto".
4. Los arrays de datos del dashboard (`index.html`).
